# Stage 7: Timing and complexity benchmark

Thesis: *Explainable AI for Early Diabetes Prediction* (CDC BRFSS 2015). Config v1.0.

Implements Ch3 3.6.2 and the *Computational* row of Table 3.2 under the protocol fixed in decision **S16** (confirmed 2026-10-16), with S8 (UTF-8) and S10 (environment guard first). Search time already exists (Stage 3); Stage 6 times are operational only.

1. **Final training time**: each frozen configuration refitted on all 202,944 development rows (`sklearn.base.clone` of the frozen pipeline; SVC including its 5-fold sigmoid calibration), 3 repeats, median and range. Each refit's test probabilities are compared with `test_predictions.csv`; the refits are then discarded. The frozen pipelines are never changed.
2. **Inference time**: `predict_proba` of the frozen pipeline on the 50,736 test rows, 1 warm-up + 5 timed repeats (KNN 3), median, ms per 1,000 rows.
3. **Explanation time, like for like**: SHAP in one process with the Stage 6 settings on the first 20 rows of the 500-row core (KNN 5), median seconds per row; LIME on the first 5 panel cases, seed 1, median seconds per explanation.
4. **Complexity descriptors** read from the frozen pipelines. No composite score (3.6.4); memory is not measured (3.6.2).

**Run only after Stage 6 has finished**, in a fresh kernel, on mains power, with no other heavy programs open (timings are affected by anything else running). Expected time on the PC: 15-25 minutes.

In [1]:
# Cell 1 - environment guard (S10): runs before numpy / scikit-learn are imported.
import os
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/thesis_xai"
else:
    PROJECT_DIR = os.path.abspath(".")
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import env_setup
env_setup.ensure_pinned(PROJECT_DIR, install=IN_COLAB)

All pinned packages present at the pinned versions.


{}

In [2]:
# Cell 2 - configuration (S16)
import gc
import hashlib
import json
import platform
import statistics
import time
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import shap
import sklearn
from sklearn.base import clone

import thesis_common as tc

MACHINE = "colab" if IN_COLAB else "pc"
STAGE = "stage7_timing_benchmark.ipynb"
FREEZE_SHA256 = "1dd5bf06708d148399675d9fc657857df1c86f015cbd83c24e79e8754b48c0d8"   # Stage 4 freeze (S4-047)
PRED_SHA256 = "b9813461cd03e4552b50c245c8750648d08b9c48c1960d249642982bc15114ed"     # Stage 5 predictions (S5-136)
EXPECTED = {"dev_rows": 202944, "test_rows": 50736}
MODELS = list(tc.MODELS)
FEATURES = list(tc.FEATURES)
TREE = ["DT", "RF", "XGB", "LGBM", "CAT"]
PERM = ["SVC", "KNN", "GNB", "MLP"]
TRAIN_REPEATS = 3
INFER_REPEATS = {m: (3 if m == "KNN" else 5) for m in MODELS}
SHAP_ROWS = {m: (5 if m == "KNN" else 20) for m in MODELS}
LIME_CASES = 5
MAX_EVALS, LIME_SAMPLES = 250, 5000                                   # as Stage 6 (P3, D4)
RUN_BEFORE_STAGE6_FINISHED = False                                    # leave False for the thesis run
OUT = os.path.join(PROJECT_DIR, "stage7_outputs")
os.makedirs(OUT, exist_ok=True)
S6 = os.path.join(PROJECT_DIR, "stage6_outputs")
if not os.path.exists(os.path.join(S6, "stage6_report.md")) and not RUN_BEFORE_STAGE6_FINISHED:
    raise SystemExit("Stage 6 has not finished (stage6_outputs/stage6_report.md is missing). Timings must not run "
                     "while Stage 6 is running. Run this notebook after Stage 6.")
SS14 = np.random.SeedSequence(tc.MASTER_SEED).generate_state(14)
PERM_SEED = int(np.random.SeedSequence(int(SS14[13])).generate_state(2)[0])           # Stage 6 main permutation seed
LIME_SEED = int(np.random.SeedSequence(tc.SEEDS["lime_perturbation"]).generate_state(3)[0])
print(f"machine={MACHINE}  python={platform.python_version()}  cpus={os.cpu_count()}  shap={shap.__version__}")


def row_seed(base, row_id):
    return int(np.random.SeedSequence([int(base), int(row_id)]).generate_state(1)[0])


def timed(fn):
    gc.collect()
    t0 = time.perf_counter()
    r = fn()
    return time.perf_counter() - t0, r

C:\Users\USER\Downloads\thesis_xai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


machine=pc  python=3.13.0  cpus=12  shap=0.52.0


In [3]:
# Cell 3 - freeze, pipelines, predictions, data and samples
fz_path = os.path.join(PROJECT_DIR, "stage4_outputs", "frozen_models.json")
if tc.sha256_file(fz_path) != FREEZE_SHA256:
    raise SystemExit("frozen_models.json is not the Stage 4 freeze. Restore it.")
with open(fz_path, encoding="utf-8") as f:
    FZ = json.load(f)
PIPE = {m: os.path.join(PROJECT_DIR, FZ["models"][m]["pipeline"]) for m in MODELS}
for m in MODELS:
    if tc.sha256_file(PIPE[m]) != FZ["models"][m]["pipeline_sha256"]:
        raise SystemExit(f"{m}: frozen pipeline changed. Restore it.")
pred_path = os.path.join(PROJECT_DIR, "stage5_outputs", "test_predictions.csv")
if tc.sha256_file(pred_path) != PRED_SHA256:
    raise SystemExit("test_predictions.csv is not the Stage 5 file. Restore it.")
PRED = pd.read_csv(pred_path, float_precision="round_trip")

X_all, y_all = tc.load_data(PROJECT_DIR)
idx = tc.load_indices(PROJECT_DIR)
dev_df = idx["dev_folds.csv"]
dev_rows = dev_df[next(c for c in ["row_id", "row", "idx", "index"] if c in dev_df.columns)].to_numpy()
test_rows = np.setdiff1d(np.arange(len(X_all)), dev_rows)
X_dev, y_dev = X_all.iloc[dev_rows].reset_index(drop=True), y_all[dev_rows]     # same order as Stages 3-4
X_test = X_all.iloc[test_rows].reset_index(drop=True)
assert len(X_dev) == EXPECTED["dev_rows"] and len(X_test) == EXPECTED["test_rows"]
assert np.array_equal(PRED.row_id.to_numpy(), test_rows)


def ids(name):
    d = idx[name]
    c = [c for c in d.columns if pd.api.types.is_integer_dtype(d[c]) and d[c].is_unique and d[c].nunique() > 2]
    return np.sort(d[c[0]].to_numpy())


BG, BG100, CORE = ids("shap_background_500_dev.csv"), ids("shap_perm_background_100_dev.csv"), ids("shap_perm_core_500_test.csv")
panel_path = os.path.join(S6, "case_panel.csv")
PANEL = pd.read_csv(panel_path).row_id.to_numpy()[:LIME_CASES] if os.path.exists(panel_path) else CORE[:LIME_CASES]
XBG, XBG100 = X_all.iloc[BG].reset_index(drop=True), X_all.iloc[BG100].reset_index(drop=True)
print(f"development {len(X_dev):,} rows, test {len(X_test):,} rows; SHAP rows from the core, LIME cases from the Stage 6 panel")

development 202,944 rows, test 50,736 rows; SHAP rows from the core, LIME cases from the Stage 6 panel


In [4]:
# Cell 4 - complexity descriptors (read from the frozen pipelines; no composite score)
def mb(path):
    return os.path.getsize(path) / 2 ** 20


cx = []
def add(m, d, v):
    cx.append({"model": m, "descriptor": d, "value": v})


for m in MODELS:
    pipe = joblib.load(PIPE[m])
    add(m, "pipeline file size (MB)", round(mb(PIPE[m]), 3))
    if m == "SVC":
        cal = pipe.calibrated_classifiers_
        svs = [c.estimator.named_steps["clf"] for c in cal]
        add(m, "calibrated copies (sigmoid)", len(cal))
        add(m, "non-zero coefficients per copy (mean)", float(np.mean([np.count_nonzero(s.coef_) for s in svs])))
        add(m, "parameters (copies x (coefficients + intercept) + 2 per calibrator)",
            int(sum(s.coef_.size + 1 for s in svs) + 2 * len(cal)))
        continue
    clf = pipe.named_steps["clf"]
    if m == "LR":
        add(m, "non-zero coefficients", int(np.count_nonzero(clf.coef_)))
        add(m, "parameters (coefficients + intercept)", int(clf.coef_.size + 1))
    elif m == "DT":
        add(m, "depth", int(clf.get_depth())); add(m, "leaves", int(clf.get_n_leaves())); add(m, "nodes", int(clf.tree_.node_count))
    elif m == "RF":
        d = [e.get_depth() for e in clf.estimators_]; lv = [e.get_n_leaves() for e in clf.estimators_]
        add(m, "trees", len(clf.estimators_)); add(m, "depth mean", float(np.mean(d))); add(m, "depth max", int(max(d)))
        add(m, "leaves total", int(sum(lv))); add(m, "nodes total", int(sum(e.tree_.node_count for e in clf.estimators_)))
    elif m == "XGB":
        t = clf.get_booster().trees_to_dataframe()
        add(m, "trees", int(t.Tree.nunique())); add(m, "leaves total", int((t.Feature == "Leaf").sum()))
        add(m, "max_depth (setting)", int(clf.get_params()["max_depth"]))
    elif m == "LGBM":
        info = clf.booster_.dump_model()["tree_info"]
        add(m, "trees", len(info)); add(m, "leaves total", int(sum(i["num_leaves"] for i in info)))
        add(m, "num_leaves (setting)", int(clf.get_params()["num_leaves"]))
    elif m == "CAT":
        dpt = int(clf.get_all_params()["depth"])
        add(m, "trees (symmetric)", int(clf.tree_count_)); add(m, "depth", dpt); add(m, "leaves total", int(clf.tree_count_ * 2 ** dpt))
    elif m == "KNN":
        add(m, "stored training rows", int(clf.n_samples_fit_)); add(m, "predictors", int(clf.n_features_in_))
        add(m, "k (n_neighbors)", int(clf.n_neighbors)); add(m, "metric", f"minkowski p={clf.p}, weights={clf.weights}")
    elif m == "GNB":
        add(m, "parameters (2 classes x 21 x (mean, variance) + 2 priors)", int(clf.theta_.size + clf.var_.size + clf.class_prior_.size))
    elif m == "MLP":
        layers = [clf.coefs_[0].shape[0]] + [w.shape[1] for w in clf.coefs_]
        add(m, "architecture (units per layer)", "-".join(map(str, layers)))
        add(m, "trainable parameters", int(sum(w.size for w in clf.coefs_) + sum(b.size for b in clf.intercepts_)))
        add(m, "training epochs (early stopping)", int(clf.n_iter_))
    del pipe
CX = pd.DataFrame(cx)
CX.to_csv(os.path.join(OUT, "complexity.csv"), index=False, encoding="utf-8")
print(CX.to_string(index=False))

model                                                          descriptor                          value
   LR                                             pipeline file size (MB)                          0.002
   LR                                               non-zero coefficients                             21
   LR                               parameters (coefficients + intercept)                             22
   DT                                             pipeline file size (MB)                          0.051
   DT                                                               depth                             10
   DT                                                              leaves                            693
   DT                                                               nodes                           1385
   RF                                             pipeline file size (MB)                         34.659
   RF                                                  

In [5]:
# Cell 5 - final training time: refit each frozen configuration on all development rows, 3 repeats (S16b)
T, REPRO = [], []
for m in ["GNB", "DT", "LR", "SVC", "KNN", "XGB", "LGBM", "MLP", "CAT", "RF"]:
    frozen = joblib.load(PIPE[m])
    secs = []
    for r_ in range(TRAIN_REPEATS):
        est = clone(frozen)                              # same configuration, unfitted (SVC: same 5 frozen folds)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            s, _ = timed(lambda: est.fit(X_dev, y_dev))
        secs.append(s)
    if m == "RF":
        est.set_params(clf__n_jobs=1)                    # S12e: bit-stable prediction for the comparison only
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        p = est.predict_proba(X_test)[:, 1]
    d = float(np.max(np.abs(p - PRED[f"p_{m}"].to_numpy())))
    REPRO.append({"model": m, "max_abs_diff_vs_stage5": d, "identical": d == 0.0})
    T.append({"model": m, "measure": "final training time", "unit": "s", "repeats": TRAIN_REPEATS,
              "median": statistics.median(secs), "min": min(secs), "max": max(secs)})
    print(f"{m:5s} fit median {statistics.median(secs):7.2f} s (range {min(secs):.2f}-{max(secs):.2f}); "
          f"refit vs Stage 5 probabilities: {'identical' if d == 0 else f'max diff {d:.2e}'}", flush=True)
    del frozen, est
pd.DataFrame(REPRO).to_csv(os.path.join(OUT, "refit_reproduction.csv"), index=False, float_format="%.6g")

GNB   fit median    0.08 s (range 0.08-0.08); refit vs Stage 5 probabilities: identical
DT    fit median    0.76 s (range 0.73-0.76); refit vs Stage 5 probabilities: identical
LR    fit median    1.24 s (range 1.24-1.24); refit vs Stage 5 probabilities: identical
SVC   fit median    4.95 s (range 4.92-4.96); refit vs Stage 5 probabilities: identical
KNN   fit median    0.06 s (range 0.06-0.06); refit vs Stage 5 probabilities: identical
XGB   fit median    4.94 s (range 4.87-5.00); refit vs Stage 5 probabilities: identical
LGBM  fit median    3.02 s (range 2.99-3.29); refit vs Stage 5 probabilities: identical
MLP   fit median   14.57 s (range 14.32-14.69); refit vs Stage 5 probabilities: identical
CAT   fit median   14.32 s (range 14.23-14.56); refit vs Stage 5 probabilities: identical
RF    fit median    8.35 s (range 8.32-8.38); refit vs Stage 5 probabilities: identical


In [6]:
# Cell 6 - inference time on the 50,736 test rows (the consistent batch), frozen threading (S16c)
for m in MODELS:
    pipe = joblib.load(PIPE[m])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        pipe.predict_proba(X_test)                                       # warm-up (not timed)
        secs = [timed(lambda: pipe.predict_proba(X_test))[0] for _ in range(INFER_REPEATS[m])]
    k = 1000 / len(X_test) * 1000                                        # seconds per batch -> ms per 1,000 rows
    T.append({"model": m, "measure": "inference time per 1,000 rows", "unit": "ms", "repeats": len(secs),
              "median": statistics.median(secs) * k, "min": min(secs) * k, "max": max(secs) * k})
    print(f"{m:5s} predict_proba on {len(X_test):,} rows: median {statistics.median(secs):.3f} s "
          f"({statistics.median(secs) * k:.2f} ms per 1,000 rows)", flush=True)
    del pipe

LR    predict_proba on 50,736 rows: median 0.012 s (0.25 ms per 1,000 rows)
DT    predict_proba on 50,736 rows: median 0.008 s (0.16 ms per 1,000 rows)
RF    predict_proba on 50,736 rows: median 0.235 s (4.63 ms per 1,000 rows)
SVC   predict_proba on 50,736 rows: median 0.064 s (1.27 ms per 1,000 rows)
KNN   predict_proba on 50,736 rows: median 54.686 s (1077.85 ms per 1,000 rows)
GNB   predict_proba on 50,736 rows: median 0.022 s (0.44 ms per 1,000 rows)
XGB   predict_proba on 50,736 rows: median 0.103 s (2.02 ms per 1,000 rows)
LGBM  predict_proba on 50,736 rows: median 0.254 s (5.01 ms per 1,000 rows)
CAT   predict_proba on 50,736 rows: median 0.009 s (0.18 ms per 1,000 rows)
MLP   predict_proba on 50,736 rows: median 0.042 s (0.84 ms per 1,000 rows)


In [7]:
# Cell 7 - explanation time, like for like (S16d): one process, Stage 6 settings
for m in MODELS:
    pipe = joblib.load(PIPE[m])
    rows = CORE[:SHAP_ROWS[m]]
    Xr = X_all.iloc[rows].reset_index(drop=True)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if m in TREE:
            clf = pipe.named_steps["clf"]
            obj = clf.get_booster() if m == "XGB" else clf
            c_s, ex = timed(lambda: shap.TreeExplainer(obj, shap.maskers.Independent(XBG, max_samples=len(XBG)),
                                                         feature_perturbation="interventional", model_output="raw"))
            per = [timed(lambda: ex(Xr.iloc[[i]], check_additivity=False))[0] for i in range(len(Xr))]
        elif m == "LR":
            prep, clf = pipe.named_steps["prep"], pipe.named_steps["clf"]
            Zbg = prep.transform(XBG)
            c_s, ex = timed(lambda: shap.LinearExplainer(clf, shap.maskers.Independent(Zbg, max_samples=len(Zbg))))
            per = [timed(lambda: ex(prep.transform(Xr.iloc[[i]])))[0] for i in range(len(Xr))]
        else:
            def f(Z, pipe=pipe):
                U, inv = np.unique(Z, axis=0, return_inverse=True)
                return pipe.predict_proba(pd.DataFrame(U, columns=FEATURES))[:, 1][inv.ravel()]
            c_s, ex = timed(lambda: shap.PermutationExplainer(f, shap.maskers.Independent(XBG100.to_numpy(), max_samples=len(XBG100))))
            Xn, per = Xr.to_numpy(), []
            for i in range(len(Xn)):
                np.random.seed(row_seed(PERM_SEED, rows[i]))
                per.append(timed(lambda: ex(Xn[i][None, :], max_evals=MAX_EVALS, silent=True))[0])
    T.append({"model": m, "measure": "SHAP time per row", "unit": "s", "repeats": len(per),
              "median": statistics.median(per), "min": min(per), "max": max(per)})
    T.append({"model": m, "measure": "SHAP explainer set-up", "unit": "s", "repeats": 1, "median": c_s, "min": c_s, "max": c_s})
    print(f"{m:5s} SHAP: median {statistics.median(per):.4f} s per row over {len(per)} rows (set-up {c_s:.2f} s)", flush=True)
    del pipe

from lime.lime_tabular import LimeTabularExplainer
c_s, LE = timed(lambda: LimeTabularExplainer(X_dev.to_numpy(), feature_names=FEATURES,
                                             categorical_features=[i for i, f in enumerate(FEATURES) if f not in tc.NON_BINARY],
                                             discretize_continuous=True, discretizer="quartile", mode="classification",
                                             class_names=["0", "1"], random_state=0))
T.append({"model": "all", "measure": "LIME explainer set-up", "unit": "s", "repeats": 1, "median": c_s, "min": c_s, "max": c_s})
for m in MODELS:
    pipe = joblib.load(PIPE[m])
    if m == "RF":
        pipe.set_params(clf__n_jobs=1)                       # as in Stage 6
    fn = lambda Z, pipe=pipe: pipe.predict_proba(pd.DataFrame(Z, columns=FEATURES))
    per = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for r_ in PANEL:
            rs = np.random.RandomState(LIME_SEED); LE.random_state = rs; LE.discretizer.random_state = rs
            x = X_all.iloc[int(r_)].to_numpy().astype(float)
            per.append(timed(lambda: LE.explain_instance(x, fn, labels=(1,), num_features=len(FEATURES), num_samples=LIME_SAMPLES))[0])
    T.append({"model": m, "measure": "LIME time per explanation", "unit": "s", "repeats": len(per),
              "median": statistics.median(per), "min": min(per), "max": max(per)})
    print(f"{m:5s} LIME: median {statistics.median(per):.3f} s per explanation over {len(per)} cases", flush=True)
    del pipe

LR    SHAP: median 0.0024 s per row over 20 rows (set-up 0.00 s)
DT    SHAP: median 0.0024 s per row over 20 rows (set-up 0.00 s)
RF    SHAP: median 1.7920 s per row over 20 rows (set-up 0.25 s)
SVC   SHAP: median 0.1029 s per row over 20 rows (set-up 0.00 s)
KNN   SHAP: median 8.1329 s per row over 5 rows (set-up 0.00 s)
GNB   SHAP: median 0.0396 s per row over 20 rows (set-up 0.00 s)
XGB   SHAP: median 0.1620 s per row over 20 rows (set-up 0.25 s)
LGBM  SHAP: median 0.1421 s per row over 20 rows (set-up 1.15 s)
CAT   SHAP: median 0.1368 s per row over 20 rows (set-up 0.36 s)
MLP   SHAP: median 0.0519 s per row over 20 rows (set-up 0.00 s)
LR    LIME: median 0.026 s per explanation over 5 cases
DT    LIME: median 0.025 s per explanation over 5 cases
RF    LIME: median 0.173 s per explanation over 5 cases
SVC   LIME: median 0.041 s per explanation over 5 cases
KNN   LIME: median 5.917 s per explanation over 5 cases
GNB   LIME: median 0.026 s per explanation over 5 cases
XGB   LIME: med

In [8]:
# Cell 8 - tables, report and environment log
TIM = pd.DataFrame(T)
TIM.to_csv(os.path.join(OUT, "timing_benchmark.csv"), index=False, float_format="%.6g", encoding="utf-8")
S3 = os.path.join(PROJECT_DIR, "stage3_outputs", "stage3_summary.csv")
search = pd.read_csv(S3).set_index("model")["search_min"] if os.path.exists(S3) else pd.Series(dtype=float)
wide = TIM[TIM.model != "all"].pivot(index="model", columns="measure", values="median").reindex(MODELS)
wide.insert(0, "search time (min, Stage 3)", search.reindex(MODELS))
wide.to_csv(os.path.join(OUT, "table32_computational.csv"), float_format="%.6g", encoding="utf-8")
pd.set_option("display.width", 220)
print(wide.round(4).to_string())

env = tc.environment(MACHINE)
with open(os.path.join(OUT, f"env_log_stage7_{MACHINE}.json"), "w", encoding="utf-8") as f:
    json.dump({**env, "stage": STAGE, "written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "pins_differ": {k: list(v) for k, v in tc.check_pins(PROJECT_DIR).items()},
               "freeze_sha256": FREEZE_SHA256, "test_predictions_sha256": PRED_SHA256,
               "protocol": {"train_repeats": TRAIN_REPEATS, "infer_repeats": INFER_REPEATS, "shap_rows": SHAP_ROWS,
                            "lime_cases": LIME_CASES, "threading": "as frozen (RF, XGB, LGBM, CAT, KNN, MLP all cores)"}},
              f, indent=2, ensure_ascii=False)
rep = pd.DataFrame(REPRO)
L = ["# Stage 7 report: timing and complexity benchmark", "",
     f"Generated by `{STAGE}` on {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC, machine {MACHINE} ({os.cpu_count()} logical CPUs), "
     f"Python {env['python']}. Protocol S16: medians; training {TRAIN_REPEATS} repeats; inference 1 warm-up + 5 repeats (KNN 3) "
     f"on {len(X_test):,} test rows; SHAP in one process on core rows (KNN 5, others 20); LIME on {LIME_CASES} panel cases, seed 1.", "",
     "| Model | Search (min) | Training (s) | Inference (ms / 1,000 rows) | SHAP (s / row) | LIME (s / explanation) |",
     "| --- | --- | --- | --- | --- | --- |"]
for m, r in wide.iterrows():
    L.append(f"| {m} | {r['search time (min, Stage 3)']:.2f} | {r['final training time']:.2f} | {r['inference time per 1,000 rows']:.2f} | "
             f"{r['SHAP time per row']:.4f} | {r['LIME time per explanation']:.3f} |")
L += ["", f"Refits reproducing the Stage 5 probabilities exactly: {int(rep.identical.sum())}/{len(rep)}"
      + ("" if rep.identical.all() else " (others: " + ", ".join(f"{r.model} {r.max_abs_diff_vs_stage5:.1e}" for r in rep.itertuples() if not r.identical) + ")"),
      "", "Complexity descriptors: `complexity.csv`. Timings depend on this machine; they compare models, not hardware."]
with open(os.path.join(OUT, "stage7_report.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(L) + "\n")
print("\n".join(L))
print("\nSend back: everything in stage7_outputs/ (6 files).")

measure  search time (min, Stage 3)  LIME time per explanation  SHAP explainer set-up  SHAP time per row  final training time  inference time per 1,000 rows
model                                                                                                                                                       
LR                             3.79                     0.0262                 0.0003             0.0024               1.2392                         0.2461
DT                             0.37                     0.0247                 0.0046             0.0024               0.7554                         0.1573
RF                            17.72                     0.1734                 0.2497             1.7920               8.3452                         4.6286
SVC                            3.71                     0.0411                 0.0004             0.1029               4.9475                         1.2686
KNN                            4.01                     5.